Final Project Code

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os 
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By

from time import sleep

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
os.environ['Email'],os.environ['Password']

('dhruvilad1001@gmail.com', 'Littlepanda@10')

In [ ]:
driver = webdriver.Chrome()

In [ ]:
driver.get('https://www.linkedin.com/login')

In [ ]:
email = driver.find_element(By.ID,'username')
email.send_keys(os.environ['Email'])

password = driver.find_element(By.ID,'password')
password.send_keys(os.environ['Password'])

password.submit()

In [ ]:
# url = "https://www.linkedin.com/in/savaliya-mukund27/"
# url = "https://www.linkedin.com/in/mitkumar-patel-031917266/"
# url = "https://www.linkedin.com/in/dhruvi-lad-7a900b352/"
# url = "https://www.linkedin.com/in/yoganshijadav/"
# url = "https://www.linkedin.com/in/shahanavaz-diwan/"
url = "https://www.linkedin.com/in/daksha-thakkar-79a98577/"
# url = "https://www.linkedin.com/in/pankaj-sonagara/"
driver.get(url)

In [14]:
profile_data = {}

In [ ]:
page_source = driver.page_source
soup = BeautifulSoup(page_source,'lxml')

name = soup.find("h1")
if name:
    print(name.get_text().strip())
else:
    print("Name not found")

profile_data['name'] = name.get_text(strip=True) if name else None
profile_data['url'] = url

Yoganshi Jadav


In [16]:
profile_data

{'name': 'Yoganshi Jadav', 'url': 'https://www.linkedin.com/in/yoganshijadav/'}

In [17]:
headline = soup.find('div',{'class': 'text-body-medium break-words'})
headline = headline.get_text().strip()

profile_data['headline'] = headline

In [18]:
profile_data

{'name': 'Yoganshi Jadav',
 'url': 'https://www.linkedin.com/in/yoganshijadav/',
 'headline': 'B.Tech Final Year Student | Web Developer | ASP.NET | SQL | Python Enthusiast'}

About

In [19]:
from selenium.common.exceptions import NoSuchElementException

try:
    driver.find_element(By.CLASS_NAME,"inline-show-more-text__button").click()
except NoSuchElementException:
    pass  # Button not present → ignore

page_source = driver.page_source
soup = BeautifulSoup(page_source, 'lxml')

about_div = soup.find('div', {'class': 'display-flex ph5 pv3'})
if not about_div:  # fallback: sometimes it's in a <section>
    about_div = soup.find('section', {'id': 'about'})

if about_div:
    about = about_div.get_text().strip()
    profile_data['about'] = about if about else None
else:
    profile_data['about'] = None


In [20]:
profile_data

{'name': 'Yoganshi Jadav',
 'url': 'https://www.linkedin.com/in/yoganshijadav/',
 'headline': 'B.Tech Final Year Student | Web Developer | ASP.NET | SQL | Python Enthusiast',
 'about': 'A passionate and goal-oriented fresher with hands-on experience in developing web applications using Python, ASP.NET Framework, ASP.NET Core MVC and SQL Server. Eager to contribute to impactful IT projects, enhance technical proficiency, and grow into a capable full-stack developer within a dynamic and innovative organization.A passionate and goal-oriented fresher with hands-on experience in developing web applications using Python, ASP.NET Framework, ASP.NET Core MVC and SQL Server. Eager to contribute to impactful IT projects, enhance technical proficiency, and grow into a capable full-stack developer within a dynamic and innovative organization.'}

Experience

In [ ]:
import json, re
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def get_experience_tokens(driver, timeout=15):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.35);")

    try:
        exp_section = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.XPATH, "//section[.//span[text()='Experience']]")
            )
        )
    except:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, "//span[text()='Experience']"))
        )
        exp_section = driver.find_element(By.XPATH, "//span[text()='Experience']/ancestor::section[1]")

    exp_html = exp_section.get_attribute("outerHTML")
    exp_soup = BeautifulSoup(exp_html, "lxml")

    tokens = [s.get_text(strip=True) for s in exp_soup.select("span.visually-hidden") if s.get_text(strip=True)]
    return tokens


def parse_experience_tokens(tokens):
    if tokens and tokens[0].strip().lower() == "experience":
        tokens = tokens[1:]

    roles_set = {
        "Full-time","Part-time","Internship","Contract","Self-employed",
        "Freelance","Apprenticeship","Volunteer","Temporary","Seasonal"
    }

    def is_duration(s: str) -> bool:
        s2 = s.lower()
        return (' to ' in s2 and ('present' in s2 or 'mo' in s2 or 'yr' in s2)) or (' yr' in s2) or (' mo' in s2)

    def is_location(s: str) -> bool:
        if is_duration(s): return False
        if s in roles_set: return False
        loc_keywords = ['india','remote','on-site','hybrid','onsite','work from home']
        if any(k in s.lower() for k in loc_keywords):
            return True
        return (',' in s and len(s.split()) >= 2)

    def is_meta(s: str) -> bool:
        return s in roles_set or is_duration(s) or is_location(s) or s.lower() in ('on-site','remote','hybrid')

    i, n = 0, len(tokens)
    result = []

    while i < n:
        lookahead = tokens[i:i+12]
        role_count = sum(1 for t in lookahead if t in roles_set)

        if role_count >= 2:
            company_name = tokens[i]; i += 1
            company_duration = None

            while i < n and (is_duration(tokens[i]) or tokens[i].lower() in ('on-site','remote','hybrid')):
                if is_duration(tokens[i]) and company_duration is None:
                    company_duration = tokens[i]
                i += 1

            entry = {'company_name': company_name, 'duration': company_duration, 'designations': []}

            while i < n:
                while i < n and is_meta(tokens[i]):
                    i += 1
                if i >= n: break

                designation = tokens[i]; i += 1
                role = tokens[i] if i < n and tokens[i] in roles_set else None
                if role is not None: i += 1
                dur = tokens[i] if i < n and is_duration(tokens[i]) else None
                if dur is not None: i += 1
                loc = tokens[i] if i < n and is_location(tokens[i]) else None
                if loc is not None: i += 1

                entry['designations'].append({
                    'designation': designation,
                    **({'role': role} if role else {}),
                    **({'duration': dur} if dur else {}),
                    **({'location': loc} if loc else {}),
                    'projects': ''
                })

                window = tokens[i:i+6]
                if sum(1 for t in window if t in roles_set) == 0 and any(is_duration(t) for t in window):
                    break

            result.append(entry)

        else:
            designation = tokens[i]; i += 1
            org = tokens[i] if i < n else None
            if org is not None: i += 1
            dur = tokens[i] if i < n and is_duration(tokens[i]) else None
            if dur is not None: i += 1
            loc = tokens[i] if i < n and is_location(tokens[i]) else None
            if loc is not None: i += 1

            entry = {
                'company_name': designation,
                'duration': org,
                'designations': [{
                    'designation': designation,
                    **({'duration': dur} if dur else {}),
                    **({'location': loc} if loc else {}),
                    'projects': ''
                }]
            }
            result.append(entry)

    return result

tokens = get_experience_tokens(driver)
experience_struct = parse_experience_tokens(tokens)

profile_data["experience"] = experience_struct

In [ ]:
profile_data

certificate

In [35]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup
import time

def scrape_licenses_and_certifications(driver, profile_data):
    """
    Scrapes licenses and certifications from LinkedIn profile.
    Handles both cases: when 'Show all' button exists and when it doesn't.
    """
    
    try:
        # Wait for the page to load
        WebDriverWait(driver, 10).wait(
            EC.presence_of_element_located((By.TAG_NAME, "main"))
        )
        
        # Try to find and click "Show all licenses & certifications" button
        try:
            show_all_button = WebDriverWait(driver, 5).wait(
                EC.element_to_be_clickable((By.ID, "navigation-index-see-all-licenses-and-certifications"))
            )
            print("Found 'Show all' button. Clicking it...")
            driver.execute_script("arguments[0].click();", show_all_button)
            time.sleep(3)  # Wait for new page to load
            print("Successfully clicked 'Show all' button")
            
        except (TimeoutException, NoSuchElementException):
            print("'Show all' button not found. Proceeding with current page data...")
            # Continue with the current page - no button click needed
        
        # Get the page source after potential navigation
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'lxml')
        
        # Look for licenses section with multiple possible selectors
        licenses_section = None
        
        # Try different selectors for the licenses section
        possible_selectors = [
            'section[data-section="licensesAndCertifications"]',
            'section.artdeco-card.pb3',
            'section.pv-profile-section.pv-accomplishments-section.pv-licenses-and-certifications-section',
            'div[data-section="licensesAndCertifications"]',
            'section:has(h2:contains("Licenses & certifications"))',
        ]
        
        for selector in possible_selectors:
            try:
                if 'contains' in selector:
                    # Handle the special case with text search
                    sections = soup.find_all('section')
                    for section in sections:
                        h2 = section.find('h2')
                        if h2 and 'licenses' in h2.get_text().lower():
                            licenses_section = section
                            break
                else:
                    licenses_section = soup.select_one(selector)
                
                if licenses_section:
                    print(f"Found licenses section using selector: {selector}")
                    break
            except Exception as e:
                continue
        
        # If still not found, try a more general approach
        if not licenses_section:
            print("Trying general approach to find licenses section...")
            # Look for any section containing license/certification related text
            all_sections = soup.find_all('section')
            for section in all_sections:
                section_text = section.get_text().lower()
                if any(keyword in section_text for keyword in ['license', 'certification', 'credential']):
                    licenses_section = section
                    print("Found licenses section using general text search")
                    break
        
        if not licenses_section:
            print("No licenses & certifications section found on this profile")
            profile_data['licenses'] = []
            return profile_data
        
        # Extract license items from the section
        items = []
        
        # Try different item selectors
        item_selectors = [
            'li.pvs-list__paged-list-item',
            'li.pv-accomplishment-entity',
            'div.pv-accomplishment-entity',
            'li[data-view-name="accomplishment-item"]',
            'div.pvs-list__item',
            'li.artdeco-list__item'
        ]
        
        for selector in item_selectors:
            items = licenses_section.select(selector)
            if items:
                print(f"Found {len(items)} license items using selector: {selector}")
                break
        
        # If no items found with specific selectors, try a more general approach
        if not items:
            print("Trying to find license items with general approach...")
            # Look for any list items or divs that might contain license data
            items = licenses_section.find_all(['li', 'div'], recursive=True)
            # Filter items that likely contain license information
            filtered_items = []
            for item in items:
                item_text = item.get_text().strip()
                if len(item_text) > 20 and any(keyword in item_text.lower() for keyword in ['institute', 'organization', 'issued', 'credential']):
                    filtered_items.append(item)
            items = filtered_items[:10]  # Limit to avoid false positives
            print(f"Found {len(items)} potential license items using general approach")
        
        def extract_license_data(item):
            """Extract license data from a single item"""
            item_dict = {}
            
            # Method 1: Try to find visually hidden spans (common in LinkedIn)
            spans = item.find_all('span', {'class': 'visually-hidden'})
            if spans and len(spans) >= 2:
                if len(spans) > 0:
                    item_dict['name'] = spans[0].get_text().strip()
                if len(spans) > 1:
                    item_dict['institute'] = spans[1].get_text().strip()
                if len(spans) > 2:
                    item_dict['issued_date'] = spans[2].get_text().strip()
                return item_dict
            
            # Method 2: Try to find specific elements by common patterns
            try:
                # Look for title/name
                title_selectors = [
                    'h3',
                    'h4', 
                    '.pv-accomplishment-entity__title',
                    '[data-field="title"]',
                    'span[aria-hidden="true"]'
                ]
                
                for selector in title_selectors:
                    title_elem = item.select_one(selector)
                    if title_elem and len(title_elem.get_text().strip()) > 3:
                        item_dict['name'] = title_elem.get_text().strip()
                        break
                
                # Look for institute/organization
                institute_selectors = [
                    '.pv-accomplishment-entity__subtitle',
                    '.pv-accomplishment-entity__detail-subtitle',
                    '[data-field="subtitle"]',
                    'p'
                ]
                
                for selector in institute_selectors:
                    institute_elem = item.select_one(selector)
                    if institute_elem and 'institute' in institute_elem.get_text().lower():
                        item_dict['institute'] = institute_elem.get_text().strip()
                        break
                
                # Look for date
                date_selectors = [
                    '.pv-accomplishment-entity__date',
                    '[data-field="date"]',
                    'time'
                ]
                
                for selector in date_selectors:
                    date_elem = item.select_one(selector)
                    if date_elem:
                        item_dict['issued_date'] = date_elem.get_text().strip()
                        break
                
            except Exception as e:
                print(f"Error extracting license data: {e}")
            
            # Method 3: Fallback - extract from text content with patterns
            if not item_dict.get('name'):
                text_content = item.get_text()
                lines = [line.strip() for line in text_content.split('\n') if line.strip()]
                
                if len(lines) >= 1:
                    item_dict['name'] = lines[0]
                if len(lines) >= 2:
                    # Look for institute in subsequent lines
                    for line in lines[1:]:
                        if any(keyword in line.lower() for keyword in ['institute', 'organization', 'university', 'academy']):
                            item_dict['institute'] = line
                            break
                    
                    # Look for date patterns
                    for line in lines:
                        if any(keyword in line.lower() for keyword in ['issued', 'earned', '202', '201']):
                            item_dict['issued_date'] = line
                            break
            
            return item_dict if item_dict.get('name') else None
        
        # Extract data from all items
        item_list = []
        for item in items:
            try:
                license_data = extract_license_data(item)
                if license_data and license_data.get('name'):
                    item_list.append(license_data)
            except Exception as e:
                print(f"Error processing license item: {e}")
                continue
        
        # Remove duplicates
        seen_names = set()
        unique_licenses = []
        for license_item in item_list:
            name = license_item.get('name', '').lower()
            if name and name not in seen_names:
                seen_names.add(name)
                unique_licenses.append(license_item)
        
        profile_data['licenses'] = unique_licenses
        print(f"Successfully extracted {len(unique_licenses)} licenses/certifications")
        
        # Print extracted data for verification
        for i, license_item in enumerate(unique_licenses, 1):
            print(f"\nLicense {i}:")
            for key, value in license_item.items():
                print(f"  {key}: {value}")
        
    except Exception as e:
        print(f"Error scraping licenses and certifications: {e}")
        profile_data['licenses'] = []
    
    return profile_data

# Example usage - Remove the triple quotes to run this code:
if __name__ == "__main__":
    # Initialize your driver and profile_data dictionary first
    driver = webdriver.Chrome()  # or your preferred driver
    profile_data = {}
    
    # Navigate to the LinkedIn profile page
    driver.get("https://linkedin.com/in/yoganshijadav/")
    
    # Call the function
    profile_data = scrape_licenses_and_certifications(driver, profile_data)
    
    # Print results
    print(f"\nFinal result: {len(profile_data.get('licenses', []))} licenses found")
    for license_item in profile_data.get('licenses', []):
        print(license_item)
    
    # Go back to previous page if needed
    # driver.back()

Error scraping licenses and certifications: 'WebDriverWait' object has no attribute 'wait'

Final result: 0 licenses found


In [34]:
profile_data

{'name': 'Yoganshi Jadav',
 'url': 'https://www.linkedin.com/in/yoganshijadav/',
 'headline': 'B.Tech Final Year Student | Web Developer | ASP.NET | SQL | Python Enthusiast',
 'about': 'A passionate and goal-oriented fresher with hands-on experience in developing web applications using Python, ASP.NET Framework, ASP.NET Core MVC and SQL Server. Eager to contribute to impactful IT projects, enhance technical proficiency, and grow into a capable full-stack developer within a dynamic and innovative organization.A passionate and goal-oriented fresher with hands-on experience in developing web applications using Python, ASP.NET Framework, ASP.NET Core MVC and SQL Server. Eager to contribute to impactful IT projects, enhance technical proficiency, and grow into a capable full-stack developer within a dynamic and innovative organization.',
 'licenses': []}

TRY Code 

In [2]:
#-------------------------Basic_Info

import warnings
warnings.filterwarnings("ignore")

import os 
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By

from time import sleep

from dotenv import load_dotenv
load_dotenv()

os.environ['Email'],os.environ['Password']

driver = webdriver.Chrome()

driver.get('https://www.linkedin.com/login')

email = driver.find_element(By.ID,'username')
email.send_keys(os.environ['Email'])

password = driver.find_element(By.ID,'password')
password.send_keys(os.environ['Password'])

password.submit()

# url = "https://www.linkedin.com/in/savaliya-mukund27/"
# url = "https://www.linkedin.com/in/mitkumar-patel-031917266/"
# url = "https://www.linkedin.com/in/dhruvi-lad-7a900b352/"
# url = "https://www.linkedin.com/in/yoganshijadav/"
# url = "https://www.linkedin.com/in/shahanavaz-diwan/"
url = "https://www.linkedin.com/in/daksha-thakkar-79a98577/"
# url = "https://www.linkedin.com/in/pankaj-sonagara/"
# url = "https://www.linkedin.com/in/sonagara-alpaben2411/"
driver.get(url)

profile_data = {}

page_source = driver.page_source
soup = BeautifulSoup(page_source,'lxml')

name = soup.find("h1")
if name:
    print(name.get_text().strip())
else:
    print("Name not found")

profile_data['name'] = name.get_text(strip=True) if name else None
profile_data['url'] = url

profile_data

headline = soup.find('div',{'class': 'text-body-medium break-words'})
headline = headline.get_text().strip()

profile_data['headline'] = headline

profile_data

#-------------------------About

from selenium.common.exceptions import NoSuchElementException

try:
    driver.find_element(By.CLASS_NAME,"inline-show-more-text__button").click()
except NoSuchElementException:
    pass  # Button not present → ignore

page_source = driver.page_source
soup = BeautifulSoup(page_source, 'lxml')

about_div = soup.find('div', {'class': 'display-flex ph5 pv3'})
if not about_div:  # fallback: sometimes it's in a <section>
    about_div = soup.find('section', {'id': 'about'})

if about_div:
    about = about_div.get_text().strip()
    profile_data['about'] = about if about else None
else:
    profile_data['about'] = None

profile_data

#-------------------------Experience

import re, json
from bs4 import BeautifulSoup
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By


def get_experience_tokens(driver, timeout=15):
    """Scroll and fetch all hidden text tokens from the Experience section"""
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.35);")

    try:
        exp_section = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.XPATH, "//section[.//span[text()='Experience']]")
            )
        )
    except:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, "//span[text()='Experience']"))
        )
        exp_section = driver.find_element(
            By.XPATH, "//span[text()='Experience']/ancestor::section[1]"
        )

    exp_html = exp_section.get_attribute("outerHTML")
    exp_soup = BeautifulSoup(exp_html, "lxml")

    tokens = [
        s.get_text(strip=True)
        for s in exp_soup.select("span.visually-hidden")
        if s.get_text(strip=True)
    ]
    return tokens


def parse_experience_tokens(tokens):
    """Parse tokens into clean company_name, designation, duration, location"""
    if tokens and tokens[0].strip().lower() == "experience":
        tokens = tokens[1:]

    roles_set = {
        "Full-time", "Part-time", "Internship", "Contract", "Self-employed",
        "Freelance", "Apprenticeship", "Volunteer", "Temporary", "Seasonal"
    }

    def is_duration(s: str) -> bool:
        s2 = s.lower()
        return (" to " in s2 and ("present" in s2 or "mo" in s2 or "yr" in s2)) or (" yr" in s2) or (" mo" in s2)

    def is_location(s: str) -> bool:
        if is_duration(s): return False
        if s in roles_set: return False
        loc_keywords = ["india", "remote", "on-site", "hybrid", "onsite", "work from home"]
        if any(k in s.lower() for k in loc_keywords):
            return True
        return ("," in s and len(s.split()) >= 2)

    def clean_company(s: str) -> str:
        """Remove role-type suffixes but keep the company name"""
        if not s:
            return ""
        for r in roles_set:
            s = s.replace("· " + r, "").replace(r, "")
        return s.strip()

    def looks_like_description(s: str, is_company=False) -> bool:
        # Company names are always preserved
        if is_company:
            return False
        return (len(s.split()) > 15 or len(s) > 120 or s.lower().endswith((".png", ".jpg", ".jpeg")))

    result = []
    i, n = 0, len(tokens)

    while i < n:
        designation, company, dur, loc = "", "", "", ""

        if i < n:
            designation = tokens[i]; i += 1

        if i < n:
            if tokens[i] not in roles_set and not is_duration(tokens[i]) and not is_location(tokens[i]):
                company = tokens[i]; i += 1

        if i < n and is_duration(tokens[i]):
            dur = tokens[i]; i += 1

        if i < n and is_location(tokens[i]):
            loc = tokens[i]; i += 1

        # Fix: swap if designation looks like duration
        if is_duration(designation):
            dur, designation = designation, ""

        # Skip junk designations, but never skip companies
        if looks_like_description(designation):
            designation = ""

        # 🚨 Ensure company is always kept if it contains company-like keywords
        if company and any(kw in company.lower() for kw in ["pvt", "inc", "solutions", "system", "hr", "ltd"]):
            pass  # keep as company
        elif looks_like_description(company, is_company=True):
            company = ""

        # Skip completely empty
        if not designation and not company:
            continue

        # Append cleaned record
        result.append({
            "company_name": clean_company(company),
            "designation": designation,
            "duration": dur,
            "location": loc
        })

    # Final filter: drop junk durations
    clean_result = [
        r for r in result
        if not looks_like_description(r["duration"])
    ]

    return clean_result


# ---------- Usage ----------
if __name__ == "__main__":
    # Step 1: Get tokens from LinkedIn page (driver must be logged in & opened profile)
    tokens = get_experience_tokens(driver)

    # Step 2: Parse into clean structure
    experience_struct = parse_experience_tokens(tokens)

    # Step 3: Store into profile_data
    # profile_data = {}
    profile_data["experience"] = experience_struct

    # Step 4: Print JSON for verification
    # print(json.dumps(profile_data["experience"], indent=4, ensure_ascii=False))

profile_data

#-------------------------Education (FIXED - Now gets ALL education entries)

try:
    # First try to scroll to education section and get all data
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.5);")
    sleep(2)
    
    # Try to find and click "see all education" button if it exists
    try:
        see_all_education_btn = driver.find_element(By.ID, "navigation-index-see-all-education")
        see_all_education_btn.click()
        sleep(3)
        print("✅ Clicked 'see all education' button")
        clicked_see_all_education = True
    except NoSuchElementException:
        print("ℹ️ 'See all education' button not found, scraping from main profile page")
        clicked_see_all_education = False
    
    # Get page source and parse
    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'lxml')
    
    # Find education section - try multiple selectors
    education_data = []
    
    if clicked_see_all_education:
        # We're on the dedicated education page
        edu_section = soup.find('section', {'class': 'artdeco-card pb3'})
        if edu_section:
            edu_blocks = edu_section.find_all("li", {"class": "pvs-list__paged-list-item"})
        else:
            edu_blocks = soup.find_all("li", {"class": "pvs-list__paged-list-item"})
    else:
        # We're on the main profile page
        try:
            edu_section = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//section[.//span[text()='Education']]"))
            )
            edu_html = edu_section.get_attribute("outerHTML")
            edu_soup = BeautifulSoup(edu_html, "lxml")
            edu_blocks = edu_soup.find_all("li", {"class": "artdeco-list__item"})
            
            # If no blocks found, try alternative selector
            if not edu_blocks:
                edu_blocks = edu_soup.find_all("li", {"class": "pvs-list__paged-list-item"})
        except:
            edu_blocks = []
    
    # Process all education blocks
    for block in edu_blocks:
        spans = block.find_all("span", {"class": "visually-hidden"})
        if not spans:
            continue

        edu_dict = {}
        # Get all available span data
        for i, span in enumerate(spans):
            text = span.get_text(strip=True)
            if i == 0:
                edu_dict["college"] = text
            elif i == 1:
                edu_dict["degree"] = text
            elif i == 2:
                edu_dict["duration"] = text
            elif i == 3:
                edu_dict["grade"] = text
            elif i == 4 and "grade" not in edu_dict:
                edu_dict["grade"] = text

        # Only add if we have at least college name
        if edu_dict.get("college"):
            education_data.append(edu_dict)

    profile_data["education"] = education_data
    print(f"✅ Found {len(education_data)} education entries")
    
    # Go back if we clicked see all education
    if clicked_see_all_education:
        driver.back()
        sleep(2)

except Exception as e:
    print(f"❌ Error in education section: {e}")
    profile_data["education"] = []

profile_data

#-------------------------Certificate (FIXED - Now handles both cases: few and many certificates)

try:
    # First scroll to licenses section
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.7);")
    sleep(2)
    
    # Try to find "see all licenses" button
    try:
        see_all_licenses_btn = driver.find_element(By.ID, "navigation-index-see-all-licenses-and-certifications")
        see_all_licenses_btn.click()
        sleep(3)
        print("✅ Clicked 'see all licenses' button")
        clicked_see_all_licenses = True
    except NoSuchElementException:
        print("ℹ️ 'See all licenses' button not found, scraping from main profile page")
        clicked_see_all_licenses = False

    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'lxml')

    item_list = []

    if clicked_see_all_licenses:
        # We're on the dedicated licenses page
        licenses_section = soup.find('section', {'class': 'artdeco-card pb3'})
        if licenses_section:
            items = licenses_section.find_all('li', {'class': 'pvs-list__paged-list-item'})
        else:
            # Fallback: find all license items on the page
            items = soup.find_all('li', {'class': 'pvs-list__paged-list-item'})
    else:
        # We're on the main profile page - look for licenses section
        try:
            # Find the licenses section on main profile
            licenses_section = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//section[.//span[contains(text(), 'Licenses') or contains(text(), 'Certifications')]]"))
            )
            licenses_html = licenses_section.get_attribute("outerHTML")
            licenses_soup = BeautifulSoup(licenses_html, "lxml")
            items = licenses_soup.find_all("li", {"class": "artdeco-list__item"})
            
            # If no items found, try alternative selector
            if not items:
                items = licenses_soup.find_all("li", {"class": "pvs-list__paged-list-item"})
        except:
            # Last fallback - search entire page for license-like content
            items = soup.find_all('li', {'class': 'artdeco-list__item'})

    def get_license(item):
        spans = item.find_all('span', {'class': 'visually-hidden'})
        item_dict = {}

        # Filter out spans that contain connection/profile info
        filtered_spans = []
        for span in spans:
            text = span.get_text().strip().lower()
            if not any(keyword in text for keyword in ['connection', 'mutual', 'profile', 'view', 'connect']):
                filtered_spans.append(span.get_text().strip())

        if len(filtered_spans) > 0:
            item_dict['name'] = filtered_spans[0]
        if len(filtered_spans) > 1:
            item_dict['institute'] = filtered_spans[1]
        if len(filtered_spans) > 2:
            item_dict['issued_date'] = filtered_spans[2]

        return item_dict

    # Process all license items
    for item in items:
        license_data = get_license(item)
        # Only add if we have at least a name
        if license_data.get('name'):
            item_list.append(license_data)

    profile_data['licenses'] = item_list
    print(f"✅ Found {len(item_list)} license/certification entries")
    
    # Go back if we clicked see all licenses
    if clicked_see_all_licenses:
        driver.back()
        sleep(2)

except Exception as e:
    print(f"❌ Error in licenses section: {e}")
    profile_data['licenses'] = []

profile_data

#-------------------------Project (FIXED - Added error handling)

try:
    # Try to find and click "see all projects" button
    try:
        see_all_projects_btn = driver.find_element(By.ID, "navigation-index-see-all-projects")
        see_all_projects_btn.click()
        sleep(3)
        print("✅ Clicked 'see all projects' button")
        clicked_see_all_projects = True
    except NoSuchElementException:
        print("ℹ️ 'See all projects' button not found, scraping from main profile page")
        clicked_see_all_projects = False

    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'lxml')

    if clicked_see_all_projects:
        # We're on the dedicated projects page
        projects_section = soup.find('section', {'class': 'artdeco-card pb3'})
        if projects_section:
            items = projects_section.find_all('li', {'class': 'pvs-list__paged-list-item'})
        else:
            items = soup.find_all('li', {'class': 'pvs-list__paged-list-item'})
    else:
        # Try to find projects on main profile page
        try:
            projects_section = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//section[.//span[text()='Projects']]"))
            )
            projects_html = projects_section.get_attribute("outerHTML")
            projects_soup = BeautifulSoup(projects_html, "lxml")
            items = projects_soup.find_all("li", {"class": "artdeco-list__item"})
            if not items:
                items = projects_soup.find_all("li", {"class": "pvs-list__paged-list-item"})
        except:
            items = []

    def get_project(item): 
        spans = item.find_all('span', {'class': 'visually-hidden'})
        item_dict = {}

        # Filter out connection/profile related spans
        filtered_spans = []
        for span in spans:
            text = span.get_text().strip().lower()
            if not any(keyword in text for keyword in ['connection', 'mutual', 'profile', 'view']):
                filtered_spans.append(span.get_text().strip())

        if len(filtered_spans) > 0:
            item_dict['project_name'] = filtered_spans[0]
        if len(filtered_spans) > 1:
            item_dict['duration'] = filtered_spans[1]
        if len(filtered_spans) > 2:
            item_dict['description'] = filtered_spans[2]
        if len(filtered_spans) > 3:
            item_dict['skills'] = filtered_spans[3]

        return item_dict

    item_list = []
    for item in items:
        project_data = get_project(item)
        if project_data.get('project_name'):
            item_list.append(project_data)

    profile_data['projects'] = item_list
    print(f"✅ Found {len(item_list)} project entries")
    
    # Go back if we clicked see all projects
    if clicked_see_all_projects:
        driver.back()
        sleep(2)

except Exception as e:
    print(f"❌ Error in projects section: {e}")
    profile_data['projects'] = []

profile_data

#-------------------------Courses (FIXED - Added error handling)

try:
    # Scroll to find courses section
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.8);")
    sleep(2)

    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'lxml')

    # Try to find courses section
    courses_list = []
    
    try:
        # Look for courses in the main profile
        courses_section = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, "//section[.//span[contains(text(), 'Course')]]"))
        )
        courses_html = courses_section.get_attribute("outerHTML")
        courses_soup = BeautifulSoup(courses_html, "lxml")
        items = courses_soup.find_all("li", {"class": "artdeco-list__item"})
        if not items:
            items = courses_soup.find_all("li", {"class": "pvs-list__paged-list-item"})
    except:
        # Fallback - search for any course-like items
        items = soup.find_all('li', {'class': 'pvs-list__paged-list-item'})

    for item in items:
        spans = item.find_all('span', {'class': 'visually-hidden'})
        
        # Filter relevant spans
        filtered_spans = []
        for span in spans:
            text = span.get_text().strip().lower()
            if not any(keyword in text for keyword in ['connection', 'mutual', 'profile']):
                filtered_spans.append(span.get_text().strip())
        
        if len(filtered_spans) >= 2:  # At least course name and associated with
            item_dict = {}
            item_dict['course_name'] = filtered_spans[0] if len(filtered_spans) > 0 else ""
            item_dict['associated_with'] = filtered_spans[1] if len(filtered_spans) > 1 else ""
            courses_list.append(item_dict)

    profile_data["courses"] = courses_list
    print(f"✅ Found {len(courses_list)} course entries")

except Exception as e:
    print(f"❌ Error in courses section: {e}")
    profile_data["courses"] = []

profile_data

#-------------------------Awards (FIXED - Added error handling)

try:
    # Try to find and click "see all honors and awards" button
    try:
        see_all_awards_btn = driver.find_element(By.ID, "navigation-index-see-all-honorsandawards")
        see_all_awards_btn.click()
        sleep(3)
        print("✅ Clicked 'see all honors and awards' button")
        clicked_see_all_awards = True
    except NoSuchElementException:
        print("ℹ️ 'See all honors and awards' button not found, scraping from main profile page")
        clicked_see_all_awards = False

    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'lxml')

    awards_list = []

    if clicked_see_all_awards:
        # We're on the dedicated awards page
        awards_section = soup.find('section', {'class': 'artdeco-card pb3'})
        if awards_section:
            award_cards = awards_section.find_all('li', {'class': 'pvs-list__paged-list-item'})
        else:
            award_cards = soup.find_all('li', {'class': 'pvs-list__paged-list-item'})
    else:
        # Try to find awards on main profile page
        try:
            awards_section = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//section[.//span[contains(text(), 'Honors') or contains(text(), 'Awards')]]"))
            )
            awards_html = awards_section.get_attribute("outerHTML")
            awards_soup = BeautifulSoup(awards_html, "lxml")
            award_cards = awards_soup.find_all("li", {"class": "artdeco-list__item"})
            if not award_cards:
                award_cards = awards_soup.find_all("li", {"class": "pvs-list__paged-list-item"})
        except:
            # Fallback
            potential_awards_sections = soup.find_all("section", {"class": "artdeco-card pv-profile-card break-words mt2"})
            award_cards = []
            for section in potential_awards_sections:
                header = section.find("h2", class_="pvs-header__title")
                if header and any("Honors & awards" in span.get_text(strip=True) for span in header.find_all("span")):
                    award_cards = section.find_all("li", {"class": "artdeco-list__item"})
                    break

    for card in award_cards:
        spans = [s.get_text(strip=True) for s in card.find_all("span", {"class": "visually-hidden"})]
        
        # Filter out connection/profile related spans
        filtered_spans = [span for span in spans 
                         if not any(keyword in span.lower() for keyword in ["connection", "studied at", "mutual", "profile"])]

        if filtered_spans:
            award_dict = {}
            if len(filtered_spans) >= 1: 
                award_dict['award_name'] = filtered_spans[0]
            if len(filtered_spans) >= 2:
                # Parse issued by and date
                parts = filtered_spans[1].split('·')
                award_dict['issued_by'] = parts[0].replace('Issued by', '').strip()
                if len(parts) > 1:
                    award_dict['issued_date'] = parts[1].strip()
            if len(filtered_spans) >= 3:
                award_dict['associated_with'] = filtered_spans[2].replace('Associated with', '').strip()

            if award_dict.get('award_name'):  # Only add if we have award name
                awards_list.append(award_dict)

    profile_data['honors_awards'] = awards_list
    print(f"✅ Found {len(awards_list)} award entries")
    
    # Go back if we clicked see all awards
    if clicked_see_all_awards:
        driver.back()
        sleep(2)

except Exception as e:
    print(f"❌ Error in awards section: {e}")
    profile_data['honors_awards'] = []

profile_data

#------------------------- name.json file

import json
import re

user_name = profile_data.get("name", "default_user")

safe_name = re.sub(r'[^a-zA-Z0-9_]', '_', user_name)

file_name = f"{safe_name}_profile_data.json"

with open(file_name, 'w') as f:
    json.dump(profile_data, f, indent=4)

print(f"JSON file created: {file_name}")

#-------------------------save data mongodb

import json
import glob
import os
from pymongo import MongoClient
from datetime import datetime

# MongoDB connection details
MONGO_URI = "mongodb://localhost:27017/"
DATABASE_NAME = "Linkedin"
COLLECTION_NAME = "profile_data"

# Folder where all JSON profile files are stored
JSON_FOLDER_PATH = r"D:\Linkdin-Profile-Scraper-In-Python"

try:
    client = MongoClient(MONGO_URI)
    db = client[DATABASE_NAME]
    collection = db[COLLECTION_NAME]

    # Step 0: Remove existing duplicates (keep one document per URL)
    pipeline = [
        {"$group": {"_id": "$url", "ids": {"$addToSet": "$_id"}, "count": {"$sum": 1}}},
        {"$match": {"count": {"$gt": 1}}}
    ]
    duplicates = list(collection.aggregate(pipeline))
    for doc in duplicates:
        ids = doc["ids"]
        ids.pop(0)  # Keep one, delete the rest
        collection.delete_many({"_id": {"$in": ids}})
    print(f"✅ Removed {len(duplicates)} duplicate URLs (if any)")

    # Step 1: Ensure unique index on URL
    collection.create_index("url", unique=True)
    print("✅ Unique index created successfully")

    # Step 2: Process JSON files
    json_files = glob.glob(os.path.join(JSON_FOLDER_PATH, "*.json"))

    def save_profile(profile):
        profile["last_updated"] = datetime.utcnow()
        collection.update_one(
            {"url": profile.get("url")},
            {"$set": profile},
            upsert=True
        )

    total = 0
    for file_path in json_files:
        with open(file_path, "r", encoding="utf-8") as file:
            file_data = json.load(file)

            if isinstance(file_data, list):
                for profile in file_data:
                    save_profile(profile)
                    total += 1
            else:
                save_profile(file_data)
                total += 1

        print(f"✅ Processed file: {os.path.basename(file_path)}")

    print(f"🎯 Total profiles processed: {total}")

except Exception as e:
    print(f"Error: {e}")
finally:
    if 'client' in locals() and client:
        client.close()

#-------------------------insert JSON → clean → export to Excel

import os
import re
import pandas as pd
from typing import Dict, List, Any
from pymongo import MongoClient

# ---------- CONFIG ----------
MONGO_URI = "mongodb://localhost:27017/"
DATABASE_NAME = "Linkedin"
COLLECTION_NAME = "profile_data"
OUTPUT_EXCEL_PATH = "cleaned_profile_data.xlsx"
# ----------------------------


# ---------- Cleaning helpers ----------
def clean_text_data(text: Any, aggressive_remove_symbols: bool = False) -> str:
    if text is None:
        return ""
    s = str(text)
    replacements = {
        'Ã¢â‚¬Â¢': '•', 'Ã‚Â·': '·', 'Ã¢â‚¬â„¢': "'", 'Ã¢â‚¬Å"': '"', 'Ã¢â‚¬': '"',
        '\u2022': '•', '\xa0': ' ', '\u200b': ''
    }
    for k, v in replacements.items():
        s = s.replace(k, v)
    s = re.sub(r'[\x00-\x1F\x7F]+', ' ', s)
    s = re.sub(r'\r\n|\r|\n', ' | ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    if aggressive_remove_symbols:
        s = re.sub(r"[^0-9A-Za-z\s\.,@:/\-\(\)\|%'\"]+", '', s)
    return s


def recursive_clean(obj: Any, aggressive_remove_symbols: bool = False) -> Any:
    if isinstance(obj, str):
        return clean_text_data(obj, aggressive_remove_symbols)
    elif isinstance(obj, dict):
        return {k: recursive_clean(v, aggressive_remove_symbols) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [recursive_clean(v, aggressive_remove_symbols) for v in obj]
    else:
        return obj
# --------------------------------------


# ---------- Duration / location extraction ----------
def extract_duration_info(duration_str: str) -> Dict[str, str]:
    if not duration_str:
        return {"duration": "", "start_date": "", "end_date": ""}
    s = duration_str.strip()
    pattern1 = r'(\w{3}\s\d{4})\s+to\s+(\w{3}\s\d{4}|\w+)\s+·\s+(.+)'
    m1 = re.search(pattern1, s)
    if m1:
        return {"duration": clean_text_data(m1.group(3)), "start_date": m1.group(1), "end_date": m1.group(2)}
    pattern2 = r'(\d+\s+(?:yr|mo)s?(?:\s+\d+\s+(?:yr|mo)s?)?)'
    m2 = re.search(pattern2, s)
    if m2:
        return {"duration": m2.group(1), "start_date": "", "end_date": ""}
    return {"duration": clean_text_data(s), "start_date": "", "end_date": ""}


def extract_location_info(location_str: str) -> Dict[str, str]:
    if not location_str:
        return {"city": "", "state": "", "country": "", "work_mode": ""}
    s = clean_text_data(location_str)
    work_mode_pattern = r'\b(Remote|On-site|Hybrid|Onsite)\b'
    wm = re.search(work_mode_pattern, s, re.IGNORECASE)
    work_mode = wm.group(1) if wm else ""
    s_clean = re.sub(work_mode_pattern, '', s, flags=re.IGNORECASE).strip()
    s_clean = re.sub(r'\s*·\s*', '', s_clean)

    parts = [p.strip() for p in s_clean.split(',') if p.strip()]
    city = parts[0] if len(parts) > 0 else ""
    state = parts[1] if len(parts) > 1 else ""
    country = parts[2] if len(parts) > 2 else (parts[-1] if parts else "")
    return {"city": city, "state": state, "country": country, "work_mode": work_mode}
# ------------------------------------------------------------------------

def process_experience_data(experience_list: List[Dict]) -> List[Dict]:
    processed_experiences = []
    if not experience_list:
        return processed_experiences
    for exp in experience_list:
        company_name = clean_text_data(exp.get('company_name', ''))
        company_duration = clean_text_data(exp.get('duration', ''))

        # if "designations" missing → fall back to flat keys
        designations = exp.get('designations')
        if not designations:
            designations = [{
                "designation": exp.get("designation", ""),
                "duration": exp.get("duration", ""),
                "location": exp.get("location", ""),
                "role": exp.get("role", ""),
                "projects": exp.get("projects", "")
            }]

        for designation in designations:
            duration_info = extract_duration_info(designation.get('duration', ''))
            location_info = extract_location_info(designation.get('location', ''))
            processed_experiences.append({
                'company_name': company_name,
                'designation': clean_text_data(designation.get('designation', '')),
                'employment_type': clean_text_data(designation.get('role', '')),
                'duration': duration_info['duration'],
                'start_date': duration_info['start_date'],
                'end_date': duration_info['end_date'],
                'city': location_info['city'],
                'state': location_info['state'],
                'country': location_info['country'],
                'work_mode': location_info['work_mode'],
                'company_duration': company_duration,
                'projects': clean_text_data(designation.get('projects', ''))
            })
    return processed_experiences


def make_grouped_rows(user_name: str, processed_experiences: List[Dict]) -> pd.DataFrame:
    """Add user name to experience data and return as DataFrame"""
    for exp in processed_experiences:
        exp['name'] = user_name
    return pd.DataFrame(processed_experiences)


def process_education_data(education_list: List[Dict], user_name: str) -> List[Dict]:
    processed = []
    for edu in education_list or []:
        d = extract_duration_info(edu.get('duration', ''))
        processed.append({
            'name': user_name,
            'college': clean_text_data(edu.get('college', '')),
            'degree': clean_text_data(edu.get('degree', '')),
            'duration': d['duration'],
            'start_date': d['start_date'],
            'end_date': d['end_date'],
            'grade': clean_text_data(edu.get('grade', ''))
        })
    return processed


def process_projects_data(projects_list: List[Dict], user_name: str) -> List[Dict]:
    processed = []
    for p in projects_list or []:
        d = extract_duration_info(p.get('duration', ''))
        skills_text = p.get('skills', '') or p.get('description', '')
        skills_match = re.search(r'Skills?:\s*([^·]+)', skills_text or '')
        skills = clean_text_data(skills_match.group(1)) if skills_match else clean_text_data(p.get('skills', ''))
        processed.append({
            'name': user_name,
            'project_name': clean_text_data(p.get('project_name', '')),
            'duration': d['duration'],
            'start_date': d['start_date'],
            'end_date': d['end_date'],
            'description': clean_text_data(p.get('description', '')),
            'skills': skills
        })
    return processed


def process_licenses_data(licenses_list: List[Dict], user_name: str) -> List[Dict]:
    processed = []
    for lic in licenses_list or []:
        issued = lic.get('issued_date', '')
        m = re.search(r'Issued\s+(\w{3}\s+\d{4})', issued or '')
        issued_clean = m.group(1) if m else clean_text_data(issued)
        processed.append({
            'name': user_name,
            'certification_name': clean_text_data(lic.get('name', '')),
            'issuing_organization': clean_text_data(lic.get('institute', '')),
            'issue_date': issued_clean
        })
    return processed


def process_profile_data(profile_data: Dict[str, Any]) -> Dict[str, pd.DataFrame]:
    df_map = {}
    user_name = clean_text_data(profile_data.get('name', ''))
    profile_url = clean_text_data(profile_data.get('url', ''))

    # Basic Info (keep url here)
    basic = {
        'name': user_name,
        'url': profile_url,
        'headline': clean_text_data(profile_data.get('headline', '')),
        'about': clean_text_data(profile_data.get('about', '')),
    }
    df_map['Basic_Info'] = pd.DataFrame([basic])

    # Other sections (with user name only)
    if profile_data.get('experience'):
        df_map['Experience'] = make_grouped_rows(user_name, process_experience_data(profile_data.get('experience', [])))

    if profile_data.get('education'):
        df_map['Education'] = pd.DataFrame(process_education_data(profile_data.get('education', []), user_name))
    if profile_data.get('projects'):
        df_map['Projects'] = pd.DataFrame(process_projects_data(profile_data.get('projects', []), user_name))
    if profile_data.get('licenses'):
        df_map['Certifications'] = pd.DataFrame(process_licenses_data(profile_data.get('licenses', []), user_name))
    if profile_data.get('honors_awards'):
        awards = []
        for a in profile_data['honors_awards']:
            awards.append({
                'name': user_name,
                'award_name': clean_text_data(a.get('award_name', '')),
                'issued_by': clean_text_data(a.get('issued_by', '')),
                'issue_date': clean_text_data(a.get('issued_date', '')),
                'associated_with': clean_text_data(a.get('associated_with', ''))
            })
        df_map['Awards'] = pd.DataFrame(awards)

    return df_map
# ------------------------------------------------------------------


# ---------- Export to Excel ----------
def export_collection_to_excel(output_path: str, aggressive_remove_symbols: bool = False):
    engine = None
    try:
        import xlsxwriter
        engine = 'xlsxwriter'
    except Exception:
        try:
            import openpyxl
            engine = 'openpyxl'
        except Exception:
            raise RuntimeError("Install xlsxwriter or openpyxl")

    client = MongoClient(MONGO_URI)
    try:
        db = client[DATABASE_NAME]
        collection = db[COLLECTION_NAME]
        documents = list(collection.find({}))
        if not documents:
            print(f"No documents found in collection '{COLLECTION_NAME}'.")
            return

        # Collect all sections into combined lists
        combined_data = {
            "Basic_Info": [],
            "Experience": [],
            "Education": [],
            "Projects": [],
            "Certifications": [],
            "Awards": []
        }

        for doc in documents:
            doc_clean = recursive_clean(doc, aggressive_remove_symbols)
            doc_clean.pop('_id', None)
            dfs = process_profile_data(doc_clean)

            for section, df in dfs.items():
                if df is not None and not df.empty:
                    combined_data[section].append(df)

        # Write each combined section to ONE sheet
        with pd.ExcelWriter(output_path, engine=engine) as writer:
            for section, df_list in combined_data.items():
                if df_list:  # if we have data for this section
                    combined_df = pd.concat(df_list, ignore_index=True)  # merge all people
                    combined_df.to_excel(writer, sheet_name=section[:31], index=False)

        print(f"✅ Exported {len(documents)} profiles into grouped sheets at '{output_path}'.")
    finally:
        client.close()
# -------------------------------------------------------------------


if __name__ == "__main__":
    export_collection_to_excel(OUTPUT_EXCEL_PATH, aggressive_remove_symbols=False)

print("✅ LinkedIn Profile Scraper completed successfully!")
print("📋 Summary of fixes applied:")
print("   1. ✅ Education section now gets ALL education entries (not just 2)")
print("   2. ✅ Certificate section handles both few and many certificates")
print("   3. ✅ Added proper error handling for missing navigation buttons")
print("   4. ✅ All sections now have fallback scraping from main profile page")
print("   5. ✅ Added driver.back() only when necessary")
print("   6. ✅ Better filtering of unwanted spans (connection info, etc.)")
print("   7. ✅ Improved data validation and cleaning")
print("\n🎯 Your scraper is now more robust and handles edge cases properly!")

Daksha Thakkar
✅ Clicked 'see all education' button
✅ Found 3 education entries
ℹ️ 'See all licenses' button not found, scraping from main profile page
✅ Found 2 license/certification entries
ℹ️ 'See all projects' button not found, scraping from main profile page
✅ Found 0 project entries
✅ Found 2 course entries
ℹ️ 'See all honors and awards' button not found, scraping from main profile page
✅ Found 8 award entries
JSON file created: Daksha_Thakkar_profile_data.json
✅ Removed 0 duplicate URLs (if any)
✅ Unique index created successfully
✅ Processed file: Daksha_Thakkar_profile_data.json
✅ Processed file: Dhruvi_Lad_profile_data.json
✅ Processed file: Mitkumar_Patel_profile_data.json
✅ Processed file: Mukund_Savaliya_profile_data.json
✅ Processed file: Pankaj_Sonagara_profile_data.json
✅ Processed file: profile_data.json
✅ Processed file: Shahanavaz_Diwan_profile_data.json
✅ Processed file: SONAGARA_ALPABEN_profile_data.json
✅ Processed file: Yoganshi_Jadav_profile_data.json
🎯 Total pr